In [ ]:
import gc
gc.collect()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from file_manager import preprocess_file_manager
from visualization_lib import folder_shower, normalize_volume
from helper import register_and_resample, sikit_to_just_data
from helper import load_NiFty_and_save_raw_data
from stat_calc import find_periods

In [ ]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = [
    'raw',
    'filling_anatomy_gaps'
]


In [ ]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

copy data to preprocess folder

In [ ]:
load_NiFty_and_save_raw_data(orginal_data_folder,file_manager,channels, filter=['3322'])
patients = file_manager.get_file_names()
#for now NO regiter _and_register

Check of Patients 

In [ ]:
# resample_channels = ['adc','dwi'],resample_to = 't2'

# def patient_registration(resample_channels = [],resample_to = 't2'):      
#     for key in resample_channels:
#                 print("ERROR ERROR")
#                 scikit_images[key] = register_and_resample(scikit_images[key],scikit_images[resample_to])  

dispaly

In [ ]:
folder_shower(file_manager,'raw',normalize_volume)

remember about allingning

<h2>Working with holes in prostate layer</h2>

In [ ]:
from anatomy_gap_fixer import find_gaps_in_anatomy
from anatomy_gap_fixer import fix_patient_anatomy

checking out outliers

In [ ]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
for outlier in outliers:
    print(outlier)
    prostate = file_manager.load_file('raw',outlier)['anatomy'] 
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    true_indices = np.where(valid_layers)[0]
    periods = find_periods(true_indices)
    print(len(periods))
    print(periods)

FIX outliers

In [ ]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
print(outliers)
start_step = 'raw'
end_step = 'raw'

for outlier in outliers:
    outlier_data = file_manager.load_file(start_step, outlier)
    outlier_new_data = fix_patient_anatomy(outlier_data)
    file_manager.save_file(end_step,outlier,outlier_new_data)

e, sections_with_prostate  = find_gaps_in_anatomy(patients,file_manager)
print(e)

Cutting pictures into correct sizes

checking size

In [ ]:
biggest_gap = (0, -1, 0)
smallest_gap = (0,99999, 0)
for patient in sections_with_prostate:
    diff = sections_with_prostate[patient][1] - sections_with_prostate[patient][0] + 1
    if diff > biggest_gap[1]:
        biggest_gap = (patient,diff,sections_with_prostate[patient])
    if diff < smallest_gap[1]:
        smallest_gap = (patient,diff,sections_with_prostate[patient])

print(biggest_gap)
print(smallest_gap)

looking for smallest picture

In [ ]:
def find_smallest_dimensions_of_3D_arrays(patients):
    smallest_dim = [9999 for x in range (0,3)]

    for patient in patients:
        data = file_manager.load_file('raw', patient)
        shape_per_channel = [data[channel].shape for channel in channels]

        all_same = len(set(shape_per_channel)) == 1
        if not all_same:
            print(f"ALERT ALERT {patient}")

        for i in range(0,3):
            dim = shape_per_channel[0][i]
            if smallest_dim[i] > dim:
                smallest_dim[i] = dim
    return smallest_dim

smallest_dim = find_smallest_dimensions_of_3D_arrays(patients)
for i in range(0,3):
    print(f"smallest dim{i} {smallest_dim[i]}")

    

two ways of centering

In [ ]:
import statistics as stats
def find_centroid_mean(figure):
    coords = np.argwhere(figure == 1)
    centroid = coords.mean(axis=0)
    return centroid

def find_figure_box(figure):
    idx = np.where(figure == 1)
    min_indices = [i.min() for i in idx]
    max_indices = [i.max() for i in idx]
    return min_indices, max_indices

def find_centroid_non_weighted(figure):
    min_indices, max_indices = find_figure_box(figure)
    return [np.mean([min_indices[i],max_indices[i]]) for i in range (0,len(max_indices))]
    

np_centroid = find_centroid_mean(prostate)
print("Np Centroid:", np_centroid)


non_weighted_centroid = find_centroid_non_weighted(prostate)
print("Np Centroid:", non_weighted_centroid)



finding maximum prostate dimentions

In [ ]:
step = 'raw'

def find_patients_max_prostate_sizes(patients,step = 'raw'):
    maximum_prostate_size = [0,0,0]
    for patient in patients:
        prostate = file_manager.load_file(step, patient)['anatomy']
        borders = find_figure_box(prostate)
        prostate_size= [ borders[1][i] - borders[0][i]+1 for i in range (0,len(borders[0]))]
        for dim, size in enumerate(prostate_size):
            if (size > maximum_prostate_size[dim]):
                maximum_prostate_size[dim] = size
    return maximum_prostate_size

maximum_prostate_size = find_patients_max_prostate_sizes(patients)
print(maximum_prostate_size)

center cropping

In [ ]:
new_size = (160,160,24)
chosen_patient = '3322'

In [ ]:
def center_crop_shift(figure, center, crop_size):
    shape = figure.shape
    start = [int(c - s//2) for c, s in zip(center, crop_size)]
    end = [start[i] + crop_size[i] for i in range(3)]

    for i in range(3):
        if start[i] < 0:
            end[i] -= start[i]
            start[i] = 0
        if end[i] > shape[i]:
            start[i] -= end[i] - shape[i]
            end[i] = shape[i]

    return figure[start[0]:end[0], start[1]:end[1], start[2]:end[2]]

In [ ]:
data = file_manager.load_file(step, chosen_patient)
center = find_centroid_non_weighted(data['anatomy'])
shifted_data = center_crop_shift(data['anatomy'],center,new_size)


In [ ]:
def show_transformation(dictionary):
    n = len(dictionary)
    plt.figure(figsize=(5*n, 5))

    for i, (name, img) in enumerate(dictionary.items(), 1):
        plt.subplot(1, n, i)
        plt.imshow(img, cmap='gray', vmin=0, vmax=1)
        plt.title(name)
        plt.axis('off')

    plt.show()

In [ ]:
data_to_show = {
    'orginal' : normalize_volume(sikit_to_just_data(file_manager.load_file_NiFty_external(orginal_data_folder,chosen_patient,channels))['anatomy'][:, :, 19]),
    'modified' : shifted_data[:,:,9],
}
show_transformation(data_to_show)

THERE IS NO PADDING!!!!

In [ ]:
step = 'raw'

for patient in patients:

    data = file_manager.load_file(step, patient)
    center = find_centroid_non_weighted(data['anatomy'])
    for channel in data:
        data[channel] = center_crop_shift(data[channel],center,new_size)
    
    file_manager.save_file('cropped',patient,data)
    

In [ ]:
folder_shower(file_manager)
folder_shower(file_manager,step = 'cropped')